In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement

In [2]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_M25.json")
gp_model.get_next_trials(max_trials=1)

{56: {'n_ci': 0.7677229162149641, 'n_it': 0.3952304404348713}}

In [3]:
def SurrogateModelOfReality(n_ci,n_it):
    y_pred = gp_model.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0]
    return np.float64(y_pred)

In [4]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [5]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    # Quasirandom Sampling Exercise
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"s1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"s2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"b1", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    X = sampler.three.QuasirandomSampler3D_func(8,Parameters_lis).T

    for array in X:
        s1 = array[0]
        s2 = array[1]
        b1 = array[2]
        n_ci = PredictorsToCaStoichs(s1,b1)
        n_it = PredictorsToIaStoichs(s2,b1)
        my_parameters = {"s1": s1, "s2": s2, "b1": b1}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(n_ci,n_it)})

    for _ in range(7): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=3)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]
            n_ci = PredictorsToCaStoichs(s1,b1)
            n_it = PredictorsToIaStoichs(s2,b1)
            result = SurrogateModelOfReality(n_ci,n_it)
            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}
            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    client._experiment.trials.pop(28)
    client._experiment.trials.pop(27)
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
17.306845866408423

Trial 1 =========================================
13.90522826902539

Trial 2 =========================================
18.143425148367818

Trial 3 =========================================
13.889390738835464

Trial 4 =========================================
18.25381206149132

Trial 5 =========================================
18.179863425027634

Trial 6 =========================================
18.2872432095352

Trial 7 =========================================
18.17057702708172

Trial 8 =========================================
13.902594733171519

Trial 9 =========================================
13.90874192255944



/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 10 =========================================
18.01872503190688

Trial 11 =========================================
18.22421409206695

Trial 12 =========================================
17.416994164561267

Trial 13 =========================================
13.818179210129802

Trial 14 =========================================
18.273206915276397

Trial 15 =========================================
13.89740657579578

Trial 16 =========================================
16.922242109476493

Trial 17 =========================================
18.228156425476964

Trial 18 =========================================
13.776141605047362

Trial 19 =========================================
18.28439312857045

Trial 20 =========================================
15.68353709135737

Trial 21 =========================================
17.706669224521672

Trial 22 =========================================
13.878871951848247

Trial 23 =========================================
18.27334965941083

Trial 24 ===

/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 66 =========================================
13.290901763227081

Trial 67 =========================================
13.891257256594571

Trial 68 =========================================
17.621515643002954

Trial 69 =========================================
13.904212559589341

Trial 70 =========================================
13.898575059907927

Trial 71 =========================================
13.793430201281806

Trial 72 =========================================
13.908734157994537

Trial 73 =========================================
18.22891459349938

Trial 74 =========================================
18.215046256524598

Trial 75 =========================================
18.262779156057167

Trial 76 =========================================
18.130723730968306

Trial 77 =========================================
18.180940262172136

Trial 78 =========================================
18.237569148208095

Trial 79 =========================================
18.180662451367358

Trial 8

/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.13/site-packages/botorch/optim/optimize.py:331: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  generated_initial_conditions = opt_inputs.get_ic_generator()(


Trial 98 =========================================
18.16468679759494

Trial 99 =========================================
13.822781209794336

[17.30684587 13.90522827 18.14342515 13.88939074 18.25381206 18.17986343
 18.28724321 18.17057703 13.90259473 13.90874192 18.01872503 18.22421409
 17.41699416 13.81817921 18.27320692 13.89740658 16.92224211 18.22815643
 13.77614161 18.28439313 15.68353709 17.70666922 13.87887195 18.27334966
 13.90689541 17.78274269 15.0081543  18.1197626  18.15707169 18.21061385
 18.2889198  18.2714059  18.09834578 18.21186453 18.12272006 18.24714789
 18.19356066 18.01513804 18.26954093 18.17910932 13.84272875 13.90426494
 17.99986109 18.22196637 18.20229213 13.91063164 13.78742949 13.88737062
 18.27279575 18.26927375 18.13061481 18.17719705 14.88829132 18.29391232
 13.90976018 13.90853994 18.29086993 13.89937674 13.90545428 13.87154577
 18.26544218 13.90985863 13.81170269 18.10964767 18.28918559 18.24963187
 13.29090176 13.89125726 17.62151564 13.90421256 13.8985

In [6]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.293912320366296
Avg = 16.601543921136503
Std = 2.0172994059719946


In [7]:
print(y_max_arr.tolist())

[17.306845866408423, 13.90522826902539, 18.143425148367818, 13.889390738835464, 18.25381206149132, 18.179863425027634, 18.2872432095352, 18.17057702708172, 13.902594733171519, 13.90874192255944, 18.01872503190688, 18.22421409206695, 17.416994164561267, 13.818179210129802, 18.273206915276397, 13.89740657579578, 16.922242109476493, 18.228156425476964, 13.776141605047362, 18.28439312857045, 15.68353709135737, 17.706669224521672, 13.878871951848247, 18.27334965941083, 13.906895410094648, 17.78274269375035, 15.008154302946577, 18.119762604599227, 18.157071694624708, 18.21061384610185, 18.28891980200282, 18.27140589537094, 18.09834577822146, 18.21186453489101, 18.12272006021986, 18.247147891240513, 18.193560663128086, 18.015138037080284, 18.269540927055306, 18.179109319541553, 13.842728749043165, 13.904264939827325, 17.99986109315132, 18.221966367644583, 18.202292126421252, 13.910631640491228, 13.787429488026262, 13.887370617225855, 18.272795748844512, 18.269273749526494, 18.130614805806598,

In [8]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M25/DataGenerated/normal_EI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)

In [9]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M25/DataGenerated/normal_EI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
print(loadeddf)
# newdf = loadeddf.drop(loadeddf.index, inplace=True)
# pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)
# print(newdf)

             0
0    18.274787
1    13.770174
2    18.076576
3    18.270455
4    17.932169
..         ...
995  13.855521
996  18.284678
997  13.790741
998  18.164687
999  13.822781

[1000 rows x 1 columns]
